In [ ]:
# A few variants of SoRL, as ablation. 
# 1. SoRL without search, |V| = 1 : we ablate away different abstract token
# 2. SoRL for fixed-budget CoT (replacing CoT with inner CoT) instead
# 3. SoRL's "recursion" is never trained to improve upon itself
#      perhaps we should encourage it to "improve" upon itself? 

# 4. An even simpler algorithm: just use a fixed budget of inner monologue to replace CoT tokens


In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import numpy as np
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from sorl.sorl_wrapper import SorlModelWrapper

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model + checkpoint
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(model_name, abstract_vocab_size_list=[128])

hf_repo_id = "Ksgk-fy/sorl_pt"
hf_filename = "qwen2.5-0.5B_gsm8k_K4_v128_i2/final.pt"
ckpt_path = hf_hub_download(repo_id=hf_repo_id, filename=hf_filename)
ckpt = torch.load(ckpt_path, map_location=device)
state_dict = ckpt["model"] if "model" in ckpt else ckpt
clean_sd = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(clean_sd)
model = model.to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)
K = 4

print(f"Device: {device} | Step: {ckpt.get('step', 'N/A')} | Epoch: {ckpt.get('epoch', 'N/A')}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Device: cpu | Step: 2805 | Epoch: 3


In [2]:
# Load GSM8K validation set
import importlib, sorl.cot_utils
importlib.reload(sorl.cot_utils)
from data.pt_dataset import get_dataset
from sorl.cot_utils import generate_with_alignment, probe_answer_logprob, find_collapse_position, render_inline_html
from IPython.display import HTML

val_ds = get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=256)
extract_fn = val_ds.extract_answer
print(f"Val set: {len(val_ds)} samples")

Val set: 1319 samples


In [ ]:
# another 2 ablation purpose SoRL trainer: 
# 1. abstract_vocab=1, num_rollouts=1, max_iterations=1 (not really demanding a new SoRL trainer class) -- this one should test whether "diverse abstract token" help with SoRL. --> we can include this in the script for run_sorl_pt.sh

# 2.
# given a prefill text input (query)
# CoT is generated, then answer is generated
# analogously, another ver. of SoRL (that replace CoT) should look like
# given a prefill text input (query)
# CoT (inner) is generated, then answer is generated

# 3.
# another ablation is to see if we can "reward" intermediate abstract tokens (not at 'max_iterations' but at multiple iterations)
# it's really as easy as 

In [3]:
# We need to remove CoT tokens, besides inserting new abstract tokens

# --- load data ---- 
from data.pt_dataset import collate_fn

samples = [val_ds[i] for i in range(4)]
batch = collate_fn(samples)
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
prompt_len = batch["prompt_len"].to(device)
B = input_ids.shape[0]

# In general, we wish to completely replace NL tokens with Abstract tokens
# This calls for an algo that gradually "internalize" NL tokens into abstract ones
# We already conduct IB attention mask, it makes distant NL tokens redundant already, but can we avoid generating NL tokens all together? 
# We'd still need the model to decide "when to generate abstract token" and "when to generate NL token"
# WHat's the simplest algo. here? 
# What if, every time we use abstract token, we'd randomly drop some NL tokens corresponding to it? 


# ---- insert mask | drop mask -----
from sorl.sorl_trainer import infer_insert_mask, expand_prompt_len, insert_tokens_with_padding, drop_tokens, replace_reasoning_with_abstract

pad_token_id = tokenizer.pad_token_id
insert_mask = infer_insert_mask(input_ids, K, attention_mask)
expanded_prompt_len = expand_prompt_len(prompt_len, insert_mask)
expanded_data, expanded_mask = insert_tokens_with_padding(input_ids, attention_mask, insert_mask, model.vocab_sizes[0], pad_token_id)

# ---- Drop NL tokens ----
# remove_prob = 0.3
# expanded_data, expanded_mask, traj_remove_1d = drop_tokens(expanded_data, expanded_mask, remove_prob, model.vocab_sizes[0])

# ---- Inner CoT replacement ---
expanded_data, expanded_mask, traj_remove_1d = replace_reasoning_with_abstract(
    expanded_data, expanded_mask, expanded_prompt_len, model.vocab_sizes[0], n_inner_cot_tokens=8, pad_token_id=pad_token_id
)

In [ ]:
# generate abstract token with inner-cot
model.generate_inner_cot(input_ids)

In [ ]:
from sorl.trainer_compress import SoRLCompressTrainer, SoRLCompressConfig

cfg = SoRLCompressConfig(remove_prob=0.3)
trainer = SoRLCompressTrainer(model, tokenizer, train_dataset, config=cfg)
trainer.train()